###**DeepDish** is a Convolutional Neural Network (CNN) designed to classify food images into **three categories**:

-  Steak  
-  Sushi  
-  Pizza  

The model is implemented using **Python** and **PyTorch**, and trained on a **subset of the Food-101 dataset**.


In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from torch import nn
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

A custom Dataset class Food3Classes was implemented to:
- Load the original Food-101 dataset
- Filter samples belonging only to the target classes
- Remap the original class indices to a new 3-class label space

In [ ]:
TARGET_CLASSES = ["pizza", "steak", "sushi"]

class Food3Classes(Dataset):
    def __init__(self, root, split='train', transform=None, download=True):

        self.food101 = datasets.Food101(root=root, split=split, download=download)

        self.class_to_idx = self.food101.class_to_idx

        self.target_indices = [self.class_to_idx[cls] for cls in TARGET_CLASSES]

        self.map_new_labels = {original: new for new, original in enumerate(self.target_indices)}

        self.samples = [
            (self.food101._image_files[idx], label)
            for idx, label in enumerate(self.food101._labels)
            if label in self.target_indices
        ]

        self.transform = transform
        self.classes = TARGET_CLASSES

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, original_label = self.samples[idx]
        img = self.food101.loader(path)


        label = self.map_new_labels[original_label]

        if self.transform:
            img = self.transform(img)

        return img, label

### Data Preprocessing & Augmentation

Different transformations were applied for training and testing to improve generalization:

 ### Training Transformations
- Random resized cropping
- Horizontal flipping
- Random rotations
- Color jitter
- Conversion to tensor
- Normalization (for transfer learning)

These augmentations help the model become more robust to variations in scale, orientation, lighting, and composition.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [ ]:
train_data = Food3Classes(root="data", split="train", transform=train_transform)
test_data = Food3Classes(root="data", split="test", transform=test_transform)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

###Model Approach 1: Custom CNN (From Scratch)

The first approach was a custom CNN inspired by the VGG architecture (`TinyVGG`), composed of:

- Three convolutional blocks:
  - Convolution layers
  - Batch normalization
  - ReLU activations
  - Max pooling
- A lightweight classifier with:
  - Adaptive average pooling
  - Fully connected layer
  - Dropout for regularization

### Training Setup
- **Loss function**: CrossEntropyLoss  
- **Optimizer**: Adam (learning rate = 0.001)  
- **Batch size**: 32  
- **Epochs**: 20  

### Observations
While the custom CNN was able to learn basic visual patterns, its performance was limited. The model struggled to extract higher-level features from complex food images, resulting in suboptimal validation accuracy and signs of underfitting.

This highlighted the difficulty of training a deep vision model from scratch with a limited dataset and config limitations

In [ ]:
class TinyVGG(nn.Module):

  def __init__(self, input_shape, output_shape):
      super().__init__()

      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape, out_channels=32, kernel_size=3, padding=1),
          nn.BatchNorm2d(32),
          nn.ReLU(),
          nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
          nn.BatchNorm2d(32),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2)
      )

      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
          nn.BatchNorm2d(64),
          nn.ReLU(),
          nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
          nn.BatchNorm2d(64),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2)
      )

      self.conv_block_3 = nn.Sequential(
        nn.Conv2d(64, 128, 3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 128, 3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
      self.classifier = nn.Sequential(
          nn.AdaptiveAvgPool2d((1,1)),
          nn.Flatten(),
          nn.Linear(in_features=128, out_features=output_shape),
          nn.Dropout(0.5)
      )

  def forward(self, x):
    return self.classifier(self.conv_block_3(self.conv_block_2(self.conv_block_1(x))))


In [ ]:
torch.manual_seed(42)
model = TinyVGG(input_shape=3, output_shape=len(TARGET_CLASSES)).to(device)
model

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_3): Sequential(
    (0

In [ ]:
def acc_fn(y_true, y_pred):
  preds = y_pred.argmax(dim=1)
  return (preds == y_true).float().mean().item()

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):

  train_loss, train_acc = 0,0

  model.train()

  for X,y in dataloader:

    X,y = X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred, y)

    train_loss += loss.item()

    train_acc += acc_fn(y_true=y, y_pred=y_pred)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  train_loss /= len(dataloader)
  train_acc /= len(dataloader)

  print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")

  return train_loss, train_acc

In [ ]:
def test_step(model, dataloader, loss_fn, device):

  test_loss, test_acc = 0,0

  model.eval()

  for X, y in dataloader:

    X,y = X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred, y)

    test_loss += loss.item()

    test_acc += acc_fn(y, y_pred)

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)

  print(f"Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%")

  return test_loss, test_acc

In [ ]:
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device):

  for epoch in range(epochs):

    print(f"Epoch: {epoch+1}")

    train_loss, train_acc = train_step(model=model, dataloader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, device=device)

    test_loss, test_acc = test_step(model=model, dataloader=test_dataloader, loss_fn=loss_fn, device=device)



In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

EPOCHS = 20

train(model= model,
      train_dataloader=train_dataloader,
      test_dataloader=test_dataloader,
      loss_fn=loss_fn,
      optimizer=optimizer,
      epochs= EPOCHS,
      device=device)

Epoch: 1
Train loss: 1.05141 | Train acc: 0.47%
Test loss: 0.92413 | Test acc: 0.58%
Epoch: 2
Train loss: 0.99364 | Train acc: 0.49%
Test loss: 0.90435 | Test acc: 0.57%
Epoch: 3
Train loss: 0.99897 | Train acc: 0.50%
Test loss: 0.88495 | Test acc: 0.58%
Epoch: 4
Train loss: 0.99399 | Train acc: 0.50%
Test loss: 0.81782 | Test acc: 0.66%
Epoch: 5
Train loss: 0.95158 | Train acc: 0.53%
Test loss: 0.81592 | Test acc: 0.66%
Epoch: 6
Train loss: 0.94562 | Train acc: 0.54%
Test loss: 0.90868 | Test acc: 0.54%
Epoch: 7
Train loss: 0.93171 | Train acc: 0.55%
Test loss: 0.86427 | Test acc: 0.60%
Epoch: 8
Train loss: 0.90880 | Train acc: 0.56%
Test loss: 0.70121 | Test acc: 0.74%
Epoch: 9
Train loss: 0.90866 | Train acc: 0.56%
Test loss: 0.64744 | Test acc: 0.79%
Epoch: 10
Train loss: 0.90756 | Train acc: 0.57%
Test loss: 0.69704 | Test acc: 0.75%
Epoch: 11
Train loss: 0.87177 | Train acc: 0.58%
Test loss: 0.69730 | Test acc: 0.72%
Epoch: 12
Train loss: 0.87375 | Train acc: 0.57%
Test loss: 0.6

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

test_transform = weights.transforms()

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform, train_transform


(ImageClassification(
     crop_size=[224]
     resize_size=[256]
     mean=[0.485, 0.456, 0.406]
     std=[0.229, 0.224, 0.225]
     interpolation=InterpolationMode.BICUBIC
 ),
 Compose(
     RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
     RandomHorizontalFlip(p=0.5)
     ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=(-0.1, 0.1))
     ToTensor()
     Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
 ))

In [ ]:
train_data = Food3Classes(root="data", split="train", transform=train_transform)
test_data = Food3Classes(root="data", split="test", transform=test_transform)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

### Model Approach 2: Transfer Learning with EfficientNet-B0

To improve performance, a transfer learning approach was adopted using EfficientNet-B0 pretrained on ImageNet.

### Strategy
- Load pretrained EfficientNet-B0 weights
- Freeze all feature extraction layers
- Replace the classifier head with a new fully connected layer for 3 classes
- Train only the final classifier layers

This allows the model to leverage rich visual features learned from millions of images, significantly improving generalization.

---

In [ ]:
def model_transfer(output_shape, device):
  weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
  model = torchvision.models.efficientnet_b0(weights=weights)

  for param in model.features.parameters():
    param.requires_grad = False

  in_features = model.classifier[1].in_features

  model.classifier = nn.Sequential(
          nn.Dropout(p=0.2),
          nn.Linear(in_features, output_shape)
      )
  return model.to(device)


In [ ]:
model_1 = model_transfer(output_shape=len(TARGET_CLASSES), device=device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(), lr=0.001)

EPOCHS = 5

train(model= model_1,
      train_dataloader=train_dataloader,
      test_dataloader=test_dataloader,
      loss_fn=loss_fn,
      optimizer=optimizer,
      epochs= EPOCHS,
      device=device)

Epoch: 1
Train loss: 0.70305 | Train acc: 0.73%
Test loss: 0.34813 | Test acc: 0.92%
Epoch: 2
Train loss: 0.44946 | Train acc: 0.86%
Test loss: 0.25459 | Test acc: 0.93%
Epoch: 3
Train loss: 0.38649 | Train acc: 0.87%
Test loss: 0.22145 | Test acc: 0.94%
Epoch: 4
Train loss: 0.34724 | Train acc: 0.88%
Test loss: 0.19645 | Test acc: 0.95%
Epoch: 5
Train loss: 0.32481 | Train acc: 0.88%
Test loss: 0.17827 | Test acc: 0.95%


Key observations:
- Faster convergence compared to the custom CNN
- Higher validation accuracy
- Better confidence in predictions
- Reduced overfitting

In [ ]:
train_data.classes

['pizza', 'steak', 'sushi']

### Saving the model

In [ ]:
MODEL_PATH = "sps_classifier.pt"

torch.save({
    "model_state_dict": model_1.state_dict(),
    "class_names": train_data.classes
}, MODEL_PATH)



In [ ]:
from google.colab import files

files.download(MODEL_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>